# Interactive GlobalFit direction-seed study

Explore the detector-level `fiber_hits` event by event. The charge threshold is interactive; MC segments are overlaid in green and never enter the data-driven seed. The fourth panel is a combined representative-coordinate 3D view; the three 2D projections are the geometrically meaningful fibre measurements.

In [6]:
from pathlib import Path
import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
import ipywidgets as widgets
from IPython.display import display
import ROOT

# Change this path to select a scattering-length branch.
INPUT_FILE = Path('/home/tlux/HK/ND280++/LFGD_Recon_Simu/Studies/LightmapsStudie/5GeV/maps/10bin_5m_0p4mm_100kPhotons/flat.root')
DETECTOR_CENTRE = np.array([0.0, 30.0, 910.0])
print(INPUT_FILE)

/home/tlux/HK/ND280++/LFGD_Recon_Simu/Studies/LightmapsStudie/5GeV/maps/10bin_5m_0p4mm_100kPhotons/flat.root


In [7]:
def read_columns(tree_name, columns):
    frame = ROOT.RDataFrame(tree_name, str(INPUT_FILE))
    arrays = frame.AsNumpy(columns)
    return {name: np.asarray(values) for name, values in arrays.items()}

fibres = read_columns('fiber_hits', ['event', 'projection', 'x', 'y', 'z', 'charge'])
mc = read_columns('mc_virtual_segments', ['event', 'detector', 'segment', 'primary_id', 'start_x', 'start_y', 'start_z', 'stop_x', 'stop_y', 'stop_z'])
events = np.unique(fibres['event']).astype(int)
print(f'{len(events)} events, {len(fibres["event"])} fibre hits, {len(mc["event"])} MC segments')

1000 events, 583207 fibre hits, 411189 MC segments


In [8]:
# Projection numbering used by GlobalFit. Each fibre measures the two listed coordinates.
VIEWS = {
    2: ('XY (Z-directed fibres)', 0, 1),
    0: ('XZ (Y-directed fibres)', 0, 2),
    1: ('YZ (X-directed fibres)', 1, 2),
}
AXIS_NAME = ['X', 'Y', 'Z']

def event_mask(columns, event):
    return columns['event'].astype(int) == int(event)

def fibre_seed(xyz, projection):
    # This exactly mirrors GlobalLightFit.cxx: Seed().
    measured = np.array([[True, False, True], [False, True, True], [True, True, False]])
    direction = np.zeros(3)
    for point, view in zip(xyz, projection.astype(int)):
        delta = point - DETECTOR_CENTRE
        for axis in range(3):
            if measured[view, axis] and abs(delta[axis]) > abs(direction[axis]):
                direction[axis] = delta[axis]
    norm = np.linalg.norm(direction)
    return direction / norm if norm else np.full(3, np.nan)

def mc_direction(mask):
    indices = np.flatnonzero(mask & (mc['detector'].astype(int) == 0))
    if not len(indices):
        return np.full(3, np.nan)
    primaries, counts = np.unique(mc['primary_id'][indices].astype(int), return_counts=True)
    primary = primaries[np.argmax(counts)]
    indices = indices[mc['primary_id'][indices].astype(int) == primary]
    indices = indices[np.argsort(mc['segment'][indices])]
    target = indices[np.flatnonzero(mc['segment'][indices].astype(int) == 10)[0]] if np.any(mc['segment'][indices].astype(int) == 10) else indices[min(9, len(indices)-1)]
    start = np.array([mc['start_x'][indices[0]], mc['start_y'][indices[0]], mc['start_z'][indices[0]]])
    stop = np.array([mc['start_x'][target], mc['start_y'][target], mc['start_z'][target]])
    delta = stop - start
    return delta / np.linalg.norm(delta) if np.linalg.norm(delta) else np.full(3, np.nan)


In [9]:
def draw_event(event=0, threshold=10.0, colour_scale='log', show_seed=True, show_mc=True):
    fm = event_mask(fibres, event) & (fibres['charge'] >= threshold)
    mm = event_mask(mc, event)
    xyz = np.column_stack([fibres['x'][fm], fibres['y'][fm], fibres['z'][fm]])
    projection = fibres['projection'][fm].astype(int)
    charge = fibres['charge'][fm]
    seed = fibre_seed(xyz, projection) if len(xyz) else np.full(3, np.nan)
    truth = mc_direction(mm)
    positive = charge[charge > 0]
    if colour_scale == 'log' and len(positive):
        norm = LogNorm(max(float(positive.min()), 1e-6), max(float(positive.max()), float(positive.min()) * 1.001))
    else:
        norm = Normalize(float(charge.min()) if len(charge) else 0, float(charge.max()) if len(charge) else 1)

    fig = plt.figure(figsize=(15, 11), constrained_layout=True)
    axes = [fig.add_subplot(2, 2, index + 1) for index in range(3)]
    axis3d = fig.add_subplot(2, 2, 4, projection='3d')
    last_scatter = None
    for axis, (view, (title, a, b)) in zip(axes, VIEWS.items()):
        selected = projection == view
        last_scatter = axis.scatter(xyz[selected, a], xyz[selected, b], c=charge[selected], norm=norm, cmap='viridis', s=32, label='fibre hits') if np.any(selected) else None
        if show_mc:
            for index in np.flatnonzero(mm & (mc['detector'].astype(int) == 0)):
                start = [mc['start_x'][index], mc['start_y'][index], mc['start_z'][index]]
                stop = [mc['stop_x'][index], mc['stop_y'][index], mc['stop_z'][index]]
                axis.plot([start[a], stop[a]], [start[b], stop[b]], color='limegreen', linewidth=1.3, alpha=.8)
        if show_seed and np.all(np.isfinite(seed)):
            extent = 1500.0
            line = np.vstack([DETECTOR_CENTRE - extent * seed, DETECTOR_CENTRE + extent * seed])
            axis.plot(line[:, a], line[:, b], color='red', linewidth=2, label='data seed')
        axis.set(title=f'{title}: {selected.sum()} fibres', xlabel=f'{AXIS_NAME[a]} [mm]', ylabel=f'{AXIS_NAME[b]} [mm]')
        axis.set_aspect('equal', adjustable='datalim')
        axis.grid(alpha=.2)
    if len(xyz):
        axis3d.scatter(xyz[:, 0], xyz[:, 1], xyz[:, 2], c=charge, norm=norm, cmap='viridis', s=18)
    if show_mc:
        for index in np.flatnonzero(mm & (mc['detector'].astype(int) == 0)):
            axis3d.plot([mc['start_x'][index], mc['stop_x'][index]], [mc['start_y'][index], mc['stop_y'][index]], [mc['start_z'][index], mc['stop_z'][index]], color='limegreen', linewidth=1)
    if show_seed and np.all(np.isfinite(seed)):
        line = np.vstack([DETECTOR_CENTRE - 1500 * seed, DETECTOR_CENTRE + 1500 * seed])
        axis3d.plot(line[:, 0], line[:, 1], line[:, 2], color='red', linewidth=2)
    axis3d.set(title=f'Combined representative positions: {len(xyz)} fibres', xlabel='X [mm]', ylabel='Y [mm]', zlabel='Z [mm]')
    if last_scatter is not None:
        fig.colorbar(last_scatter, ax=axes, shrink=.75, label='charge')
    angle = np.degrees(np.arccos(np.clip(abs(np.dot(seed, truth)), 0, 1))) if np.all(np.isfinite(seed)) and np.all(np.isfinite(truth)) else np.nan
    fig.suptitle(f'Event {event} | threshold ≥ {threshold:g} | fibres {len(xyz)} | seed {seed.round(4)} | seed–MC angle {angle:.2f}°', fontsize=13)
    plt.show()


In [10]:
event_control = widgets.SelectionSlider(options=[int(value) for value in events], description='Event', continuous_update=False, layout=widgets.Layout(width='650px'))
threshold_control = widgets.FloatSlider(value=10, min=0, max=max(100.0, float(np.percentile(fibres['charge'], 99))), step=1, description='Threshold', continuous_update=False, readout_format='.1f', layout=widgets.Layout(width='650px'))
scale_control = widgets.ToggleButtons(options=['log', 'linear'], value='log', description='Charge')
seed_control = widgets.Checkbox(value=True, description='Data seed')
mc_control = widgets.Checkbox(value=True, description='MC overlay')
controls = widgets.VBox([event_control, threshold_control, widgets.HBox([scale_control, seed_control, mc_control])])
output = widgets.interactive_output(draw_event, {'event': event_control, 'threshold': threshold_control, 'colour_scale': scale_control, 'show_seed': seed_control, 'show_mc': mc_control})
display(controls, output)

Output()

## Interactive `homo_truth` display

This uses the pre-fibre expected photon response stored in `homo_truth`. It has independent event and threshold controls and the same MC overlay.

In [11]:
homo_truth = read_columns('homo_truth', ['event', 'projection', 'x', 'y', 'z', 'charge'])
truth_events = np.unique(homo_truth['event']).astype(int)

def draw_homo_truth(event=0, threshold=10.0, colour_scale='log', show_seed=True, show_mc=True):
    global fibres
    detector_fibres = fibres
    try:
        fibres = homo_truth
        draw_event(event, threshold, colour_scale, show_seed, show_mc)
    finally:
        fibres = detector_fibres

truth_event_control = widgets.SelectionSlider(options=[int(value) for value in truth_events], description='Event', continuous_update=False, layout=widgets.Layout(width='650px'))
truth_threshold_control = widgets.FloatSlider(value=10, min=0, max=max(100.0, float(np.percentile(homo_truth['charge'], 99))), step=1, description='Threshold', continuous_update=False, readout_format='.1f', layout=widgets.Layout(width='650px'))
truth_scale_control = widgets.ToggleButtons(options=['log', 'linear'], value='log', description='Charge')
truth_seed_control = widgets.Checkbox(value=True, description='Data seed')
truth_mc_control = widgets.Checkbox(value=True, description='MC overlay')
truth_controls = widgets.VBox([truth_event_control, truth_threshold_control, widgets.HBox([truth_scale_control, truth_seed_control, truth_mc_control])])
truth_output = widgets.interactive_output(draw_homo_truth, {'event': truth_event_control, 'threshold': truth_threshold_control, 'colour_scale': truth_scale_control, 'show_seed': truth_seed_control, 'show_mc': truth_mc_control})
display(truth_controls, truth_output)

Output()

## Experimental per-view median/diameter seed

Each view uses `median charge × factor` as its own threshold. The most widely separated surviving pair defines that view's projected direction; the three projected constraints are combined into one 3D seed. This remains a notebook experiment and is not used by `global_light_fit`.

In [12]:
def median_diameter_seed(columns, event, factor=1.0):
    mask = event_mask(columns, event)
    xyz = np.column_stack([columns['x'][mask], columns['y'][mask], columns['z'][mask]])
    view = columns['projection'][mask].astype(int)
    charge = columns['charge'][mask]
    retained = np.zeros(len(charge), dtype=bool)
    pairs, cuts, constraint = {}, {}, np.zeros((3, 3))
    for projection, (_, a, b) in VIEWS.items():
        present = view == projection
        if not np.any(present): continue
        cuts[projection] = float(np.median(charge[present])) * factor
        keep = present & (charge >= cuts[projection]); retained |= keep
        index = np.flatnonzero(keep)
        if len(index) < 2: continue
        uv = xyz[index][:, [a, b]]
        distance2 = np.sum((uv[:, None] - uv[None, :])**2, axis=2)
        i, j = np.unravel_index(np.argmax(distance2), distance2.shape)
        pairs[projection] = index[[i, j]]
        delta = uv[j] - uv[i]
        if np.linalg.norm(delta) == 0: continue
        normal = np.zeros(3); normal[a], normal[b] = -delta[1], delta[0]
        normal /= np.linalg.norm(normal); constraint += np.outer(normal, normal)
    _, vectors = np.linalg.eigh(constraint); seed = vectors[:, 0]
    if seed[np.argmax(np.abs(seed))] < 0: seed *= -1
    return xyz, view, charge, retained, pairs, cuts, seed

def draw_median_diameter(event=0, factor=1.0, show_rejected=True, show_mc=True):
    xyz, view, charge, retained, pairs, cuts, seed = median_diameter_seed(fibres, event, factor)
    mm = event_mask(mc, event); truth = mc_direction(mm)
    positive = charge[retained & (charge > 0)]
    norm = LogNorm(max(float(positive.min()), 1e-6), max(float(positive.max()), float(positive.min())*1.001)) if len(positive) else Normalize(0, 1)
    fig = plt.figure(figsize=(15, 11), constrained_layout=True)
    axes = [fig.add_subplot(2, 2, i+1) for i in range(3)]
    axis3d = fig.add_subplot(2, 2, 4, projection='3d'); coloured = None
    for axis, (projection, (title, a, b)) in zip(axes, VIEWS.items()):
        present, keep = view == projection, (view == projection) & retained
        if show_rejected: axis.scatter(xyz[present & ~retained, a], xyz[present & ~retained, b], c='lightgray', s=12, alpha=.35)
        if np.any(keep): coloured = axis.scatter(xyz[keep, a], xyz[keep, b], c=charge[keep], norm=norm, cmap='viridis', s=34)
        if projection in pairs:
            pair = pairs[projection]; axis.plot(xyz[pair, a], xyz[pair, b], 'r-o', linewidth=2.2)
        if show_mc:
            for k in np.flatnonzero(mm & (mc['detector'].astype(int) == 0)):
                start=(mc['start_x'][k],mc['start_y'][k],mc['start_z'][k]); stop=(mc['stop_x'][k],mc['stop_y'][k],mc['stop_z'][k])
                axis.plot([start[a],stop[a]],[start[b],stop[b]],color='limegreen',linewidth=1.2)
        axis.set(title=f'{title}: cut {cuts.get(projection, np.nan):.2f}, kept {keep.sum()}/{present.sum()}', xlabel=f'{AXIS_NAME[a]} [mm]', ylabel=f'{AXIS_NAME[b]} [mm]'); axis.grid(alpha=.2); axis.set_aspect('equal', adjustable='datalim')
    if np.any(retained): axis3d.scatter(xyz[retained,0],xyz[retained,1],xyz[retained,2],c=charge[retained],norm=norm,cmap='viridis',s=18)
    if show_mc:
        for k in np.flatnonzero(mm & (mc['detector'].astype(int) == 0)):
            axis3d.plot([mc['start_x'][k],mc['stop_x'][k]],[mc['start_y'][k],mc['stop_y'][k]],[mc['start_z'][k],mc['stop_z'][k]],color='limegreen',linewidth=1)
    line=np.vstack([DETECTOR_CENTRE-1500*seed,DETECTOR_CENTRE+1500*seed]); axis3d.plot(line[:,0],line[:,1],line[:,2],'r-',linewidth=2.2)
    axis3d.set(title=f'Combined seed: {retained.sum()} fibres',xlabel='X [mm]',ylabel='Y [mm]',zlabel='Z [mm]')
    if coloured is not None: fig.colorbar(coloured,ax=axes,shrink=.75,label='charge')
    angle=np.degrees(np.arccos(np.clip(abs(np.dot(seed,truth)),0,1))) if np.all(np.isfinite(truth)) else np.nan
    fig.suptitle(f'Event {event} | median × {factor:.2f} | seed {seed.round(4)} | seed–MC {angle:.2f}°'); plt.show()

diameter_event=widgets.SelectionSlider(options=[int(v) for v in events],description='Event',continuous_update=False,layout=widgets.Layout(width='650px'))
diameter_factor=widgets.FloatSlider(value=1,min=.25,max=3,step=.05,description='Median factor',continuous_update=False,layout=widgets.Layout(width='650px'))
diameter_rejected=widgets.Checkbox(value=True,description='Show rejected'); diameter_mc=widgets.Checkbox(value=True,description='MC overlay')
diameter_controls=widgets.VBox([diameter_event,diameter_factor,widgets.HBox([diameter_rejected,diameter_mc])])
diameter_output=widgets.interactive_output(draw_median_diameter,{'event':diameter_event,'factor':diameter_factor,'show_rejected':diameter_rejected,'show_mc':diameter_mc})
display(diameter_controls,diameter_output)

Output()

## Suggested study procedure

Start at threshold 0, increase it until the broad light halo disappears, and watch the red data seed and the reported seed–MC angle. Repeat across events and scattering lengths by changing `INPUT_FILE` and rerunning the loading cells. MC is only an overlay and angle diagnostic; it is never used to construct the red seed.